In [1]:
"""
VesselBridgeNet — Phase B Innovation
====================================
Features:
- SCCSA Skip Connections (Filtered Background)
- EMCAD Decoder (80% fewer FLOPs)
- BiFormer-Inspired Attention Bottleneck
- DDSP Domain-Aware Style Alignment
- Two-Phase Training & Focal Tversky Loss
"""
import os, time, json
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from tqdm.auto import tqdm

# ====================================================
# 1. Configuration
# ====================================================
SEED = 42
IMG_SIZE = 512
BATCH_SIZE = 8
DATA_DIR = '/kaggle/input/datasets/aymenslimani/green-injection-retinopathy-segmentation'
SAVE_DIR = 'results/vesselbridge_mega_model'
os.makedirs(SAVE_DIR, exist_ok=True)

PHASE1_EPOCHS = 15
PHASE1_LR = 1e-3
PHASE2_EPOCHS = 35
PHASE2_LR = 5e-5
PATIENCE = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
seed_everything(SEED)

# ====================================================
# 2. Dataset & Transforms
# ====================================================
class RetinalDataset(Dataset):
    def __init__(self, csv_file, base_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.base_dir = base_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.base_dir, row['img_path'])), cv2.COLOR_BGR2RGB)
        mask = (cv2.imread(os.path.join(self.base_dir, row['vessel_path']), 0) > 127).astype(np.float32)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        mask = mask.unsqueeze(0) if isinstance(mask, torch.Tensor) else torch.from_numpy(mask).unsqueeze(0)
        return img, mask

def get_train_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=90, border_mode=cv2.BORDER_CONSTANT, p=0.6),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.4),
        A.ElasticTransform(alpha=120, sigma=6.0, p=0.3),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# ====================================================
# 3. Loss Functions
# ====================================================
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=4/3, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, logits, targets):
        inputs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        tp = (inputs * targets).sum()
        fp = (inputs * (1 - targets)).sum()
        fn = ((1 - inputs) * targets).sum()
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return (1 - tversky) ** self.gamma

class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ft = FocalTverskyLoss()
        self.bce = nn.BCEWithLogitsLoss()
        
    def forward(self, logits, targets):
        return 0.5 * self.bce(logits, targets) + 0.5 * self.ft(logits, targets)

# ====================================================
# 4. Custom Architecture (VesselBridgeNet)
# ====================================================
class SCCSA_Block(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        ca = torch.sigmoid(avg_out + max_out)
        x_ca = x * ca
        
        avg_sp = torch.mean(x_ca, dim=1, keepdim=True)
        max_sp, _ = torch.max(x_ca, dim=1, keepdim=True)
        sp_cat = torch.cat([avg_sp, max_sp], dim=1)
        sa = torch.sigmoid(self.conv(sp_cat))
        return x_ca * sa

class DepthWiseConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.pointwise(self.depthwise(x))))

class EMCAD_Block(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            DepthWiseConv(out_ch + skip_ch, out_ch),
            DepthWiseConv(out_ch, out_ch)
        )

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class BiFormerAttention_Light(nn.Module):
    def __init__(self, dim, num_heads=8, sr_ratio=4):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.q = nn.Conv2d(dim, dim, kernel_size=1)
        self.kv = nn.Conv2d(dim, dim * 2, kernel_size=1)
        self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
        self.norm = nn.LayerNorm(dim)
        self.proj = nn.Conv2d(dim, dim, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape
        q = self.q(x).reshape(B, self.num_heads, C // self.num_heads, H * W).transpose(2, 3) 
        
        x_sr = self.sr(x)
        _, _, H_sr, W_sr = x_sr.shape
        x_sr = x_sr.reshape(B, C, H_sr * W_sr).transpose(1, 2)
        x_sr = self.norm(x_sr).transpose(1, 2).view(B, C, H_sr, W_sr)
        
        kv = self.kv(x_sr).reshape(B, 2, self.num_heads, C // self.num_heads, H_sr * W_sr)
        k, v = kv[:, 0].transpose(2, 3), kv[:, 1].transpose(2, 3) 
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        
        x_out = (attn @ v).transpose(2, 3).reshape(B, C, H, W)
        return self.proj(x_out) + x

class DDSP_StyleAlignment(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.align = nn.InstanceNorm2d(channels, affine=True)
        
    def forward(self, x):
        return self.align(x)

class VesselBridgeNet(nn.Module):
    def __init__(self, out_channels=1, pretrained=True):
        super().__init__()
        resnet = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu) 
        self.enc2 = nn.Sequential(resnet.maxpool, resnet.layer1)         
        self.enc3 = resnet.layer2                                        
        self.enc4 = resnet.layer3                                        
        
        bottleneck_dim = 512
        self.proj = nn.Conv2d(1024, bottleneck_dim, 1)
        self.ddsp = DDSP_StyleAlignment(bottleneck_dim)
        self.biformer = BiFormerAttention_Light(bottleneck_dim, num_heads=8, sr_ratio=4)
        
        self.sccsa_s3 = SCCSA_Block(512)
        self.sccsa_s2 = SCCSA_Block(256)
        self.sccsa_s1 = SCCSA_Block(64)
        
        self.dec4 = EMCAD_Block(bottleneck_dim, 512, 256)
        self.dec3 = EMCAD_Block(256, 256, 128)            
        self.dec2 = EMCAD_Block(128, 64, 64)              
        self.dec1 = EMCAD_Block(64, 0, 32)                
        self.final = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        
        t = self.proj(s4)
        t = self.ddsp(t)
        t = self.biformer(t)
        
        f3 = self.sccsa_s3(s3)
        f2 = self.sccsa_s2(s2)
        f1 = self.sccsa_s1(s1)
        
        d4 = self.dec4(t, f3)
        d3 = self.dec3(d4, f2)
        d2 = self.dec2(d3, f1)
        d1 = self.dec1(d2)
        
        out = self.final(F.interpolate(d1, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False))
        return out

def freeze_encoder(model):
    for n, param in model.named_parameters():
        if 'dec' not in n and 'final' not in n:
            param.requires_grad = False
    print("🧊 Backbone Frozen (Only Decoder active)")

def unfreeze_encoder(model):
    for param in model.parameters():
        param.requires_grad = True
    print("🔥 Entire Network Unfrozen")

# ====================================================
# 5. Training Engine
# ====================================================
def evaluate_batch(preds, targets, thresh=0.5, smooth=1e-6):
    p = (torch.sigmoid(preds) > thresh).float()
    t = targets
    tp = (p * t).sum().item()
    fp = (p * (1 - t)).sum().item()
    fn = ((1 - p) * t).sum().item()
    tn = ((1 - p) * (1 - t)).sum().item()
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    sens = (tp + smooth) / (tp + fn + smooth)
    spec = (tn + smooth) / (tn + fp + smooth)
    return {'dice': dice, 'iou': iou, 'sensitivity': sens, 'specificity': spec}

def train_model(model, train_loader, val_loader, optimizer, criterion, 
                num_epochs=50, scheduler=None, save_path='best.pth', start_epoch=1, patience=10):
    scaler = GradScaler('cuda') if device.type == 'cuda' else None
    best_dice = 0.0
    patience_cnt = 0
    history = []

    for epoch in range(start_epoch, start_epoch + num_epochs):
        t0 = time.time()
        # Train
        model.train()
        ep_loss, ep_dice, ep_iou, nb = 0, 0, 0, 0
        for images, masks in tqdm(train_loader, desc=f"Ep {epoch} Train", leave=False):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            if scaler:
                with autocast('cuda'):
                    out = model(images)
                    loss = criterion(out, masks)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            else:
                out = model(images)
                loss = criterion(out, masks)
                loss.backward(); optimizer.step()
            
            m = evaluate_batch(out, masks)
            ep_loss += loss.item(); ep_dice += m['dice']; ep_iou += m['iou']; nb += 1
            
        tr = {'loss': ep_loss/nb, 'dice': ep_dice/nb}

        # Val
        model.eval()
        ep_loss, ep_dice, ep_iou, nb = 0, 0, 0, 0
        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Ep {epoch} Val", leave=False):
                images, masks = images.to(device), masks.to(device)
                out = model(images)
                loss = criterion(out, masks)
                m = evaluate_batch(out, masks)
                ep_loss += loss.item(); ep_dice += m['dice']; ep_iou += m['iou']; nb += 1
                
        vl = {'loss': ep_loss/nb, 'dice': ep_dice/nb, 'iou': ep_iou/nb}
        if scheduler: scheduler.step(vl['dice']) # For ReduceLROnPlateau

        print(f"Ep {epoch:2d} | TrL:{tr['loss']:.4f} | VL:{vl['loss']:.4f} | VDice:{vl['dice']:.4f} | VIoU:{vl['iou']:.4f} | {time.time()-t0:.0f}s")
        
        history.append({'epoch': epoch, **tr, **vl})
        
        if vl['dice'] > best_dice:
            best_dice = vl['dice']
            patience_cnt = 0
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_dice': best_dice}, save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"⏹ Early stop triggered.")
                break
                
    return history, best_dice

# ====================================================
# 6. Main Execution Pipeline
# ====================================================
if __name__ == "__main__":
    print(f"Starting VesselBridgeNet Pipeline on {device}")
    
    # 1. Datasets
    train_ds = RetinalDataset(os.path.join(DATA_DIR, 'combined_train_split.csv'), DATA_DIR, get_train_transforms())
    val_ds = RetinalDataset(os.path.join(DATA_DIR, 'combined_val_split.csv'), DATA_DIR, get_val_transforms())
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    # 2. Model & Loss
    model = VesselBridgeNet().to(device)
    criterion = HybridLoss()
    save_path = os.path.join(SAVE_DIR, 'vesselbridge_mega_best.pth')
    
    # ==========================
    # PHASE 1: Warmup Decoder
    # ==========================
    print("\n" + "="*50 + "\nPHASE 1: Decoder Warmup\n" + "="*50)
    freeze_encoder(model)
    opt1 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=PHASE1_LR)
    sch1 = optim.lr_scheduler.ReduceLROnPlateau(opt1, mode='max', factor=0.5, patience=3)
    hist1, _ = train_model(model, train_loader, val_loader, opt1, criterion, num_epochs=PHASE1_EPOCHS, scheduler=sch1, save_path=save_path, patience=PATIENCE)
    
    # Load best Phase 1 weights before unfreezing
    model.load_state_dict(torch.load(save_path, map_location=device)['model_state_dict'])
    
    # ==========================
    # PHASE 2: End-to-End Fine-tuning
    # ==========================
    print("\n" + "="*50 + "\nPHASE 2: End-to-End Fine-tuning\n" + "="*50)
    unfreeze_encoder(model)
    opt2 = optim.AdamW(model.parameters(), lr=PHASE2_LR)
    sch2 = optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', factor=0.5, patience=5)
    hist2, _ = train_model(model, train_loader, val_loader, opt2, criterion, num_epochs=PHASE2_EPOCHS, scheduler=sch2, save_path=save_path, start_epoch=PHASE1_EPOCHS+1, patience=PATIENCE)

    # ==========================
    # 7. Final Evaluation
    # ==========================
    print("\n" + "="*50 + "\nEVALUATION\n" + "="*50)
    model.load_state_dict(torch.load(save_path, map_location=device)['model_state_dict'])
    model.eval()
    
    test_ds = RetinalDataset(os.path.join(DATA_DIR, 'combined_test_split.csv'), DATA_DIR, get_val_transforms())
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Testing"):
            preds = torch.sigmoid(model(images.to(device))).cpu().numpy().flatten()
            all_preds.append(preds)
            all_targets.append(masks.numpy().flatten())
            
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    
    # Optimal Threshold Sweep
    best_thresh, best_val_dice = 0.5, 0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for images, masks in val_loader:
            preds = torch.sigmoid(model(images.to(device))).cpu().numpy().flatten()
            val_preds.append(preds)
            val_targets.append(masks.numpy().flatten())
    vp, vt = np.concatenate(val_preds), np.concatenate(val_targets)
    
    for thresh in np.arange(0.05, 0.96, 0.05):
        pb = (vp > thresh).astype(np.float32)
        tp = (pb * vt).sum(); fp = (pb * (1-vt)).sum(); fn = ((1-pb) * vt).sum()
        d = (2*tp+1e-6)/(2*tp+fp+fn+1e-6)
        if d > best_val_dice: best_val_dice, best_thresh = d, thresh
            
    print(f"✅ Optimal Threshold: {best_thresh:.2f}")
    
    # Final Metrics
    pred_opt = (all_preds > best_thresh).astype(np.float32)
    tp = (pred_opt * all_targets).sum(); fp = (pred_opt * (1-all_targets)).sum()
    fn = ((1-pred_opt) * all_targets).sum(); tn = ((1-pred_opt) * (1-all_targets)).sum()
    s = 1e-6
    
    print("\n--- FINAL TEST RESULTS ---")
    print(f"  Dice:        {(2*tp+s)/(2*tp+fp+fn+s):.4f}")
    print(f"  IoU:         {(tp+s)/(tp+fp+fn+s):.4f}")
    print(f"  Sensitivity: {(tp+s)/(tp+fn+s):.4f}")
    print(f"  Specificity: {(tn+s)/(tn+fp+s):.4f}")
    print(f"  Precision:   {(tp+s)/(tp+fp+s):.4f}")
    print(f"  AUC-ROC:     {roc_auc_score(all_targets, all_preds):.4f}")
    print(f"  AUC-PR:      {average_precision_score(all_targets, all_preds):.4f}")
    
    print("\n--- CONFUSION MATRIX ---")
    print(confusion_matrix(all_targets, pred_opt))
    
    n_pixels = len(all_targets); pos_pixels = all_targets.sum()
    print("\n--- NAIVE BASELINE (All Background) ---")
    print(f"  Accuracy: {(n_pixels - pos_pixels) / n_pixels:.4f}")
    
    print("\n✅ Done!")


Starting VesselBridgeNet Pipeline on cuda


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 170MB/s]



PHASE 1: Decoder Warmup
🧊 Backbone Frozen (Only Decoder active)


...

Ep 43 Val:   0%|          | 0/10 [00:00<?, ?it/s]

Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7a5c99e7eb60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
    Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7a5c99e7eb60>
 Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      self._shutdown_workers()  
 ^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():
^^  ^ ^ ^ ^ ^ ^^^^^^^^^^^^^^

Ep 43 | TrL:0.1464 | VL:0.1269 | VDice:0.7884 | VIoU:0.6510 | 21s
⏹ Early stop triggered.

EVALUATION


Testing:   0%|          | 0/9 [00:00<?, ?it/s]

✅ Optimal Threshold: 0.60

--- FINAL TEST RESULTS ---
  Dice:        0.7968
  IoU:         0.6623
  Sensitivity: 0.8147
  Specificity: 0.9739
  Precision:   0.7797
  AUC-ROC:     0.9786
  AUC-PR:      0.8800

--- CONFUSION MATRIX ---
[[16052501   429910]
 [  346080  1521589]]

--- NAIVE BASELINE (All Background) ---
  Accuracy: 0.8982

✅ Done!
